<a href="https://colab.research.google.com/github/sing7726/banking-loan-risk-analysis/blob/main/Banking_Loan_Risk_Analysis_SQL_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Banking Loan Risk Analysis Using SQL and Power BI

## SQL Data Preparation and Business KPI Analysis in Google Colab

This notebook is part of my data analyst resume project. The main purpose of this project is to analyze banking customer and loan data using SQL before creating the final dashboard in Power BI.

The project focuses on customer profiles, loan applications, credit score groups, income groups, approval rates, default rates, and regional loan performance.

**Tools used:** Google Colab, Python, SQLite, SQL, pandas, CSV files, and Power BI.

**Note:** The dataset used in this project is a synthetic banking dataset created for portfolio and learning purposes.


## Project Objective

The goal of this project is to show how SQL can be used in a real data analyst workflow for a financial institution.

In this notebook, the work is divided into these steps:

1. Upload and extract the banking dataset.
2. Load CSV files into Google Colab.
3. Create a SQLite database.
4. Run data-quality checks.
5. Check duplicates, missing values, invalid values, and table relationships.
6. Create analysis-ready SQL views.
7. Calculate business KPIs for loan approval, default risk, customer segments, and regional performance.
8. Export final CSV files for Power BI.


## Step 1: Import Required Libraries

In this step, the required Python libraries are imported. pandas is used to read and export CSV files, sqlite3 is used to run SQL queries, and zipfile is used to extract the uploaded project folder.


In [ ]:
import pandas as pd
import sqlite3
import zipfile
import os
from google.colab import files

## Step 2: Upload the Project ZIP File

The banking project files are stored inside a ZIP folder. This step uploads the ZIP file into Google Colab.

Upload this file when Colab asks for it:

`banking_loan_risk_analysis_day2.zip`


In [ ]:
uploaded = files.upload()

Saving banking_loan_risk_analysis_day2.zip to banking_loan_risk_analysis_day2.zip


## Step 3: Extract the ZIP File

After uploading the ZIP file, the folder is extracted so that the CSV files can be used in the notebook.


In [ ]:
# Get the uploaded ZIP file name automatically
zip_file = list(uploaded.keys())[0]

extract_folder = "banking_project"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully.")
print(os.listdir(extract_folder))

Files extracted successfully.
['customers.csv', '04_kpi_preview_queries.sql', '02_data_quality_checks.sql', 'day2_kpi_preview.csv', 'branches.csv', 'day2_row_counts.csv', '03_cleaning_and_transformation_views.sql', 'loan_analysis_view.csv', 'customer_risk_view.csv', 'day2_data_quality_results.csv', '01_create_tables.sql', 'banking_project_schema.sql', 'loans.csv', 'accounts.csv', 'banking_loan_risk_analysis.db', 'README_day2.md', 'data_dictionary.md', 'transactions.csv']


## Step 4: Load CSV Files

In this step, all CSV files are loaded into pandas DataFrames. These tables represent customers, branches, accounts, loans, and transactions.


In [ ]:
customers = pd.read_csv(f"{extract_folder}/customers.csv")
branches = pd.read_csv(f"{extract_folder}/branches.csv")
accounts = pd.read_csv(f"{extract_folder}/accounts.csv")
loans = pd.read_csv(f"{extract_folder}/loans.csv")
transactions = pd.read_csv(f"{extract_folder}/transactions.csv")

print("CSV files loaded successfully.")

CSV files loaded successfully.


## Step 5: Preview Customer Data

Before doing SQL analysis, the first few rows of the customer table are checked. This helps to understand customer-related columns such as age, income, credit score, employment status, and branch ID.


In [ ]:
customers.head()

,customer_id,age,gender,employment_status,annual_income,credit_score,branch_id,customer_since
0,1001,56,Female,Part-time,36652,579,2,2020-04-13
1,1002,69,Female,Full-time,83543,674,5,2021-01-14
2,1003,46,Male,Retired,55049,619,14,2021-05-16
3,1004,32,Female,Part-time,35964,689,3,2023-11-27
4,1005,60,Female,Self-employed,57524,494,10,2017-12-27


## Step 6: Preview Loan Data

The loan table is also previewed because this is the main table for the project. It contains loan amount, interest rate, term, application date, and loan status.


In [ ]:
loans.head()

,loan_id,customer_id,loan_type,loan_amount,interest_rate,term_months,application_date,loan_status
0,9001,1573,Mortgage,543503.56,9.94,36,2024-07-09,Approved
1,9002,1312,Personal Loan,25402.34,11.87,12,2023-08-19,Approved
2,9003,1312,Personal Loan,24868.72,15.48,60,2023-02-02,Approved
3,9004,1281,Student Loan,29914.65,8.22,24,2023-05-09,Paid Off
4,9005,1268,Student Loan,12143.48,16.31,60,2024-08-01,Approved


## Step 7: Create the SQLite Database

The CSV files are loaded into a SQLite database. This allows SQL queries to be written directly in Google Colab.

Each DataFrame is saved as a SQL table:

- `branches`
- `customers`
- `accounts`
- `loans`
- `transactions`


In [ ]:
conn = sqlite3.connect("banking_loan_risk_analysis.db")

branches.to_sql("branches", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
accounts.to_sql("accounts", conn, index=False, if_exists="replace")
loans.to_sql("loans", conn, index=False, if_exists="replace")
transactions.to_sql("transactions", conn, index=False, if_exists="replace")

print("Database created and tables loaded successfully.")

Database created and tables loaded successfully.


## Step 8: Check Row Counts

This query checks how many records are available in each table. This is the first basic data-quality check after loading the data.


In [ ]:
query = """
SELECT 'branches' AS table_name, COUNT(*) AS total_rows FROM branches
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'accounts', COUNT(*) FROM accounts
UNION ALL
SELECT 'loans', COUNT(*) FROM loans
UNION ALL
SELECT 'transactions', COUNT(*) FROM transactions;
"""

pd.read_sql_query(query, conn)

,table_name,total_rows
0,branches,15
1,customers,1000
2,accounts,1669
3,loans,852
4,transactions,8000


## Step 9: Check Duplicate IDs

This step checks whether any primary ID values are repeated. Duplicate IDs can create problems in joins, reporting, and KPI calculations.

The important ID columns checked here are customer ID, account ID, loan ID, and transaction ID.


In [ ]:
duplicate_checks = {
    "duplicate_customers": """
        SELECT customer_id, COUNT(*) AS duplicate_count
        FROM customers
        GROUP BY customer_id
        HAVING COUNT(*) > 1;
    """,

    "duplicate_accounts": """
        SELECT account_id, COUNT(*) AS duplicate_count
        FROM accounts
        GROUP BY account_id
        HAVING COUNT(*) > 1;
    """,

    "duplicate_loans": """
        SELECT loan_id, COUNT(*) AS duplicate_count
        FROM loans
        GROUP BY loan_id
        HAVING COUNT(*) > 1;
    """,

    "duplicate_transactions": """
        SELECT transaction_id, COUNT(*) AS duplicate_count
        FROM transactions
        GROUP BY transaction_id
        HAVING COUNT(*) > 1;
    """
}

for check_name, query in duplicate_checks.items():
    print(check_name)
    display(pd.read_sql_query(query, conn))

duplicate_customers


,customer_id,duplicate_count


duplicate_accounts


,account_id,duplicate_count


duplicate_loans


,loan_id,duplicate_count


duplicate_transactions


,transaction_id,duplicate_count


## Step 10: Check Missing Values in Customer Data

This query checks important customer columns for missing values. For this project, income, credit score, and branch ID are important because they are used later for customer segmentation and loan risk analysis.


In [ ]:
query = """
SELECT
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN annual_income IS NULL THEN 1 ELSE 0 END) AS missing_annual_income,
    SUM(CASE WHEN credit_score IS NULL THEN 1 ELSE 0 END) AS missing_credit_score,
    SUM(CASE WHEN branch_id IS NULL THEN 1 ELSE 0 END) AS missing_branch_id
FROM customers;
"""

pd.read_sql_query(query, conn)

,missing_customer_id,missing_annual_income,missing_credit_score,missing_branch_id
0,0,0,0,0


## Step 11: Check Missing Values in Loan Data

This query checks important loan columns for missing values. Loan amount, interest rate, and loan status are required for KPI analysis and Power BI dashboard creation.


In [ ]:
query = """
SELECT
    SUM(CASE WHEN loan_id IS NULL THEN 1 ELSE 0 END) AS missing_loan_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN loan_amount IS NULL THEN 1 ELSE 0 END) AS missing_loan_amount,
    SUM(CASE WHEN interest_rate IS NULL THEN 1 ELSE 0 END) AS missing_interest_rate,
    SUM(CASE WHEN loan_status IS NULL THEN 1 ELSE 0 END) AS missing_loan_status
FROM loans;
"""

pd.read_sql_query(query, conn)

,missing_loan_id,missing_customer_id,missing_loan_amount,missing_interest_rate,missing_loan_status
0,0,0,0,0,0


## Step 12: Check Invalid Credit Scores

Credit scores should stay within a realistic range. In this project, the valid credit score range is considered from 300 to 900.


In [ ]:
query = """
SELECT *
FROM customers
WHERE credit_score < 300 OR credit_score > 900;
"""

pd.read_sql_query(query, conn)

,customer_id,age,gender,employment_status,annual_income,credit_score,branch_id,customer_since


## Step 13: Check Invalid Loan Amounts and Interest Rates

Loan amounts and interest rates should be positive values. This query checks for incorrect loan records where loan amount or interest rate is less than or equal to zero.


In [ ]:
query = """
SELECT *
FROM loans
WHERE loan_amount <= 0 OR interest_rate <= 0;
"""

pd.read_sql_query(query, conn)

,loan_id,customer_id,loan_type,loan_amount,interest_rate,term_months,application_date,loan_status


## Step 14: Check Invalid Account Balances

This query checks whether any account has a negative current balance. This is useful for data validation before connecting account information with customer analysis.


In [ ]:
query = """
SELECT *
FROM accounts
WHERE current_balance < 0;
"""

pd.read_sql_query(query, conn)

,account_id,customer_id,account_type,current_balance,account_status


## Step 15: Check Invalid Transaction Amounts

Transaction amounts should be greater than zero. This query checks whether there are any transaction records with invalid amounts.


In [ ]:
query = """
SELECT *
FROM transactions
WHERE amount <= 0;
"""

pd.read_sql_query(query, conn)

,transaction_id,account_id,customer_id,transaction_type,amount,transaction_date


## Step 16: Check Customer and Branch Relationship

This query checks whether every customer is connected to a valid branch. This is important because branch and province information are used later for regional loan performance analysis.


In [ ]:
query = """
SELECT c.customer_id, c.branch_id
FROM customers c
LEFT JOIN branches b
    ON c.branch_id = b.branch_id
WHERE b.branch_id IS NULL;
"""

pd.read_sql_query(query, conn)

,customer_id,branch_id


## Step 17: Check Loan and Customer Relationship

This query checks whether every loan is connected to a valid customer. If a loan does not match a customer, it should not be used in customer-level risk analysis.


In [ ]:
query = """
SELECT l.loan_id, l.customer_id
FROM loans l
LEFT JOIN customers c
    ON l.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""

pd.read_sql_query(query, conn)

,loan_id,customer_id


## Step 18: Check Transaction and Account Relationship

This query checks whether every transaction is connected to a valid bank account. This validates the relationship between transactions and accounts.


In [ ]:
query = """
SELECT t.transaction_id, t.account_id
FROM transactions t
LEFT JOIN accounts a
    ON t.account_id = a.account_id
WHERE a.account_id IS NULL;
"""

pd.read_sql_query(query, conn)

,transaction_id,account_id


## Step 19: Create Customer Risk View

This SQL view prepares customer data for analysis. It adds useful categories such as age group, income group, and credit score group.

These categories make it easier to compare customers by risk level and build Power BI visuals later.


In [ ]:
query = """
DROP VIEW IF EXISTS customer_risk_view;

CREATE VIEW customer_risk_view AS
SELECT
    c.customer_id,
    c.age,
    CASE
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 35 THEN '26-35'
        WHEN c.age BETWEEN 36 AND 50 THEN '36-50'
        WHEN c.age BETWEEN 51 AND 65 THEN '51-65'
        ELSE '66+'
    END AS age_group,
    c.gender,
    c.employment_status,
    c.annual_income,
    CASE
        WHEN c.annual_income < 30000 THEN 'Low Income'
        WHEN c.annual_income BETWEEN 30000 AND 69999 THEN 'Middle Income'
        WHEN c.annual_income BETWEEN 70000 AND 119999 THEN 'High Income'
        ELSE 'Very High Income'
    END AS income_group,
    c.credit_score,
    CASE
        WHEN c.credit_score < 580 THEN 'Poor'
        WHEN c.credit_score BETWEEN 580 AND 669 THEN 'Fair'
        WHEN c.credit_score BETWEEN 670 AND 739 THEN 'Good'
        WHEN c.credit_score BETWEEN 740 AND 799 THEN 'Very Good'
        ELSE 'Excellent'
    END AS credit_score_group,
    b.city,
    b.province,
    b.branch_name
FROM customers c
LEFT JOIN branches b
    ON c.branch_id = b.branch_id;
"""

conn.executescript(query)

print("customer_risk_view created successfully.")

customer_risk_view created successfully.


## Step 20: Create Loan Analysis View

This SQL view combines loan information with customer risk information. It joins loans with the customer risk view using `customer_id`.

This view will be the main table for loan approval, loan default, customer segment, and regional performance analysis.


In [ ]:
query = """
DROP VIEW IF EXISTS loan_analysis_view;

CREATE VIEW loan_analysis_view AS
SELECT
    l.loan_id,
    l.customer_id,
    l.loan_type,
    l.loan_amount,
    l.interest_rate,
    l.term_months,
    l.application_date,
    l.loan_status,
    cr.age,
    cr.age_group,
    cr.gender,
    cr.employment_status,
    cr.annual_income,
    cr.income_group,
    cr.credit_score,
    cr.credit_score_group,
    cr.city,
    cr.province,
    cr.branch_name,
    CASE
        WHEN l.loan_status = 'Approved' THEN 1
        ELSE 0
    END AS approved_flag,
    CASE
        WHEN l.loan_status = 'Rejected' THEN 1
        ELSE 0
    END AS rejected_flag,
    CASE
        WHEN l.loan_status = 'Defaulted' THEN 1
        ELSE 0
    END AS defaulted_flag
FROM loans l
LEFT JOIN customer_risk_view cr
    ON l.customer_id = cr.customer_id;
"""

conn.executescript(query)

print("loan_analysis_view created successfully.")

loan_analysis_view created successfully.


## Step 21: Preview Final Loan Analysis View

This step previews the final joined view. This helps confirm that loan data, customer information, branch details, and risk groups are all combined correctly.


In [ ]:
query = """
SELECT *
FROM loan_analysis_view
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,loan_id,customer_id,loan_type,loan_amount,interest_rate,term_months,application_date,loan_status,age,age_group,...,annual_income,income_group,credit_score,credit_score_group,city,province,branch_name,approved_flag,rejected_flag,defaulted_flag
0,9001,1573,Mortgage,543503.56,9.94,36,2024-07-09,Approved,50,36-50,...,81538,High Income,691,Good,Quebec City,Quebec,Quebec City Main Branch,1,0,0
1,9002,1312,Personal Loan,25402.34,11.87,12,2023-08-19,Approved,65,51-65,...,68662,Middle Income,590,Fair,Surrey,British Columbia,Surrey Main Branch,1,0,0
2,9003,1312,Personal Loan,24868.72,15.48,60,2023-02-02,Approved,65,51-65,...,68662,Middle Income,590,Fair,Surrey,British Columbia,Surrey Main Branch,1,0,0
3,9004,1281,Student Loan,29914.65,8.22,24,2023-05-09,Paid Off,57,51-65,...,18807,Low Income,674,Good,Toronto,Ontario,Toronto Main Branch,0,0,0
4,9005,1268,Student Loan,12143.48,16.31,60,2024-08-01,Approved,20,18-25,...,19543,Low Income,552,Poor,Quebec City,Quebec,Quebec City Main Branch,1,0,0
5,9006,1418,Mortgage,383711.10,9.38,24,2023-07-10,Rejected,19,18-25,...,71048,High Income,629,Fair,Toronto,Ontario,Toronto Main Branch,0,1,0
6,9007,1607,Student Loan,21676.40,11.74,300,2024-02-19,Approved,21,18-25,...,78298,High Income,636,Fair,Winnipeg,Manitoba,Winnipeg Main Branch,1,0,0
7,9008,1569,Auto Loan,21469.99,13.93,36,2023-08-16,Rejected,41,36-50,...,14581,Low Income,533,Poor,Toronto,Ontario,Toronto Main Branch,0,1,0
8,9009,1828,Personal Loan,20250.99,5.59,24,2023-09-07,Approved,35,26-35,...,33110,Middle Income,694,Good,Quebec City,Quebec,Quebec City Main Branch,1,0,0
9,9010,1828,Auto Loan,35588.50,5.70,12,2023-05-11,Approved,35,26-35,...,33110,Middle Income,694,Good,Quebec City,Quebec,Quebec City Main Branch,1,0,0


## Step 22: Loan Status Summary

This query summarizes loans by status. It shows how many loans are approved, rejected, defaulted, or paid off, along with total loan amount, average loan amount, and average interest rate.

This is useful for the executive summary page in Power BI.


In [ ]:
query = """
SELECT
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amount), 2) AS total_loan_amount,
    ROUND(AVG(loan_amount), 2) AS average_loan_amount,
    ROUND(AVG(interest_rate), 2) AS average_interest_rate
FROM loans
GROUP BY loan_status
ORDER BY total_loans DESC;
"""

pd.read_sql_query(query, conn)

,loan_status,total_loans,total_loan_amount,average_loan_amount,average_interest_rate
0,Approved,544,54294143.32,99805.41,11.33
1,Rejected,149,16541264.68,111015.20,12.44
2,Paid Off,99,11124425.56,112367.93,11.33
3,Defaulted,60,7211191.11,120186.52,13.49


## Step 23: Approval Rate and Default Rate

This query calculates the main banking KPIs for the project:

- Total loan applications
- Approved loans
- Rejected loans
- Defaulted loans
- Paid off loans
- Approval rate
- Default rate

These KPIs can be used as cards in Power BI.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_applications,
    SUM(CASE WHEN loan_status = 'Approved' THEN 1 ELSE 0 END) AS approved_loans,
    SUM(CASE WHEN loan_status = 'Rejected' THEN 1 ELSE 0 END) AS rejected_loans,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,
    SUM(CASE WHEN loan_status = 'Paid Off' THEN 1 ELSE 0 END) AS paid_off_loans,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Approved' THEN 1 ELSE 0 END) / COUNT(*), 2) AS approval_rate_percent,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_percent
FROM loans;
"""

pd.read_sql_query(query, conn)

,total_applications,approved_loans,rejected_loans,defaulted_loans,paid_off_loans,approval_rate_percent,default_rate_percent
0,852,544,149,60,99,63.85,7.04


## Step 24: Loan Status by Credit Score Group

This query compares loan status across different credit score groups. It helps identify whether lower credit score groups have higher rejection or default patterns.

This is useful for credit risk analysis.


In [ ]:
query = """
SELECT
    credit_score_group,
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(AVG(loan_amount), 2) AS average_loan_amount,
    ROUND(AVG(interest_rate), 2) AS average_interest_rate
FROM loan_analysis_view
GROUP BY credit_score_group, loan_status
ORDER BY credit_score_group, total_loans DESC;
"""

pd.read_sql_query(query, conn)

,credit_score_group,loan_status,total_loans,average_loan_amount,average_interest_rate
0,Excellent,Approved,8,115183.29,4.70
1,Excellent,Paid Off,1,23242.71,4.16
2,Fair,Approved,254,93780.71,11.23
3,Fair,Rejected,54,116778.21,11.35
4,Fair,Paid Off,43,141974.16,11.23
5,Fair,Defaulted,12,138513.95,12.26
6,Good,Approved,90,106228.84,7.90
7,Good,Paid Off,21,83269.72,8.03
8,Good,Rejected,19,103280.21,8.47
9,Good,Defaulted,5,263244.36,8.40


## Step 25: Loan Status by Income Group

This query compares loan status across income groups. It helps analyze how income level may relate to approval, rejection, and default patterns.

This can be used in the customer segmentation section of the Power BI dashboard.


In [ ]:
query = """
SELECT
    income_group,
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(AVG(loan_amount), 2) AS average_loan_amount,
    ROUND(AVG(interest_rate), 2) AS average_interest_rate
FROM loan_analysis_view
GROUP BY income_group, loan_status
ORDER BY income_group, total_loans DESC;
"""

pd.read_sql_query(query, conn)

,income_group,loan_status,total_loans,average_loan_amount,average_interest_rate
0,High Income,Approved,221,92583.55,10.02
1,High Income,Rejected,37,62456.98,10.20
2,High Income,Paid Off,34,86594.64,10.82
3,High Income,Defaulted,14,124674.08,12.05
4,Low Income,Approved,67,123554.94,12.84
5,Low Income,Rejected,49,130087.45,13.51
6,Low Income,Defaulted,25,109457.00,14.20
7,Low Income,Paid Off,20,102044.75,13.16
8,Middle Income,Approved,247,100265.23,12.22
9,Middle Income,Rejected,61,121561.88,12.99


## Step 26: Regional Loan Performance

This query analyzes loan performance by province. It calculates total loans, total loan amount, average loan amount, approval rate, and default rate by region.

This is useful for finding which provinces have stronger loan activity and where default risk may be higher.


In [ ]:
query = """
SELECT
    province,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amount), 2) AS total_loan_amount,
    ROUND(AVG(loan_amount), 2) AS average_loan_amount,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Approved' THEN 1 ELSE 0 END) / COUNT(*), 2) AS approval_rate_percent,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_percent
FROM loan_analysis_view
GROUP BY province
ORDER BY total_loan_amount DESC;
"""

pd.read_sql_query(query, conn)

,province,total_loans,total_loan_amount,average_loan_amount,approval_rate_percent,default_rate_percent
0,Ontario,293,28409050.73,96959.22,60.75,5.80
1,Quebec,104,12920208.19,124232.77,62.50,5.77
2,Saskatchewan,106,12850249.51,121228.77,67.92,10.38
3,Alberta,102,10740011.46,105294.23,62.75,9.80
4,British Columbia,124,10544726.99,85038.12,63.71,5.65
5,Nova Scotia,63,7301058.26,115889.81,73.02,6.35
6,Manitoba,60,6405719.53,106761.99,66.67,8.33


## Step 27: Export Final Files for Power BI

The final SQL views are exported as CSV files. These files can be imported into Power BI to build the dashboard.

The two final files are:

- `customer_risk_view.csv`
- `loan_analysis_view.csv`


In [ ]:
customer_risk_view = pd.read_sql_query("SELECT * FROM customer_risk_view;", conn)
loan_analysis_view = pd.read_sql_query("SELECT * FROM loan_analysis_view;", conn)

customer_risk_view.to_csv("customer_risk_view.csv", index=False)
loan_analysis_view.to_csv("loan_analysis_view.csv", index=False)

print("Files exported successfully.")

Files exported successfully.


## Step 28: Download Final CSV Files

This step downloads the final Power BI-ready CSV files from Google Colab.

These files will be used in the next part of the project to create the Power BI dashboard.


In [ ]:
files.download("customer_risk_view.csv")
files.download("loan_analysis_view.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Project Summary

In this notebook, SQL was used to prepare and analyze banking loan data. The main work completed includes loading data, checking data quality, validating table relationships, creating customer risk groups, creating final analysis views, and calculating important financial KPIs.

This project shows practical data analyst skills such as SQL joins, `CASE WHEN` statements, grouping, aggregation, data validation, KPI calculation, and preparing clean data for Power BI.

The next step is to use the exported CSV files in Power BI and create an interactive dashboard for banking loan risk analysis.
